# Task 5 - Source metadata ingestion into MongoDB

Spark reads only `cpg.source-metadata.v1`, parses an explicit
schema, and writes replacement upserts keyed by `_id=file_id`. Its checkpoint
is retained on a Docker volume.

## Approach and rationale

**Approach:** One Spark Structured Streaming query reads the metadata topic with
an explicit nested schema and `read_committed` isolation. The MongoDB connector
uses replacement upserts with `_id=file_id`; Kafka offsets are stored in each
document as evidence, while Spark recovery state lives in a persistent
checkpoint volume.

**Why this approach:** Metadata is naturally one document per source file, so a
replacement upsert expresses the desired latest-state model and prevents field
fragments from surviving an update. An explicit schema surfaces incompatible
events early. The checkpoint, rather than an application-maintained offset,
lets Spark resume with its own checkpointed offset and progress protocol.

**Alternatives and trade-offs:** Append-only MongoDB writes would retain event
history but violate the required no-duplication final state. `foreachBatch`
could implement custom writes, yet the MongoDB Spark Connector already supplies
the required sink semantics with less custom code. `startingOffsets=earliest`
is useful only for an empty checkpoint; preserving the volume is what prevents
old offsets from being replayed after restart.

In [1]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import mongo_snapshot

snapshot = mongo_snapshot('867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7')
print(json.dumps(snapshot, indent=2))
assert snapshot['documents'] == snapshot['distinct_files'] == 61
assert snapshot['document']['_id'] == '867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'
print('PASS: MongoDB has one replacement-upserted document per file')

{
  "documents": 61,
  "distinct_files": 61,
  "document": {
    "_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
    "path": "optimum/version.py",
    "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
    "node_counts": {
      "AST": 25,
      "SYNTHETIC": 4
    },
    "edge_counts": {
      "AST": 24,
      "CFG": 9,
      "DFG": 3
    },
    "processed_at": "2026-07-24T15:09:20.140539Z",
    "run_id": "de2b478f74c345138e77a636e2d2d660",
    "kafka_offset": 416
  },
  "other_documents_count": 60,
  "other_documents_digest": "80b1e9f81a9a9d42c67f1195204b990cac32ae1ed60665c5a1e68814a7be5cc1"
}
PASS: MongoDB has one replacement-upserted document per file


In [2]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import checkpoint_offset, kafka_end_offset

progress = {
    'spark_checkpoint_offset': checkpoint_offset(),
    'metadata_topic_end_offset': kafka_end_offset('cpg.source-metadata.v1'),
}
print(json.dumps(progress, indent=2))
assert progress['spark_checkpoint_offset'] == progress['metadata_topic_end_offset']
print('PASS: Spark checkpoint has consumed the metadata topic')

{
  "spark_checkpoint_offset": 418,
  "metadata_topic_end_offset": 418
}
PASS: Spark checkpoint has consumed the metadata topic


![MongoDB UI capture of the replay file; the executable output above verifies the final live offset and checkpoint](figures/mongodb-ui.png)

*MongoDB UI capture of the replay file; the executable output above verifies the final live offset and checkpoint*

## Reflection

**Worked:** Spark consumes only metadata, MongoDB maintains one `_id=file_id` document per source file, and the checkpoint reaches the Kafka end offset.

**Issue encountered during development:** First startup was slow while resolving connector packages, and document counts alone could not distinguish replacement from duplication.

**Resolution:** A persistent Ivy cache and checkpoint volume support restart, while distinct-ID counts, content hashes, Kafka offsets, and replacement-upsert settings verify the sink behavior.